### 태스크 1.1: 기초 설정 및 기본 에이전트 생성

Strands Agents 프레임워크를 사용하여 고객 지원 에이전트 프로토타입을 설정합니다. 이 프로토타입은 에이전트 프로토타입에서 프로덕션용 솔루션에 이르는 전체 여정을 탐색하기 위한 출발점 역할을 합니다.

이 태스크를 완료하면 에이전트는 다음과 같은 기본 아키텍처를 갖게 됩니다.

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_ko_kr.png" width="75%"/>
</div>

**이미지 설명: 로컬 도구를 사용하여 로컬에서 실행되는 Simple Agent 프로토타입**

종속성을 설치하고 AWS SDK, AgentCore 구성 요소, Strands 프레임워크를 비롯해 필요한 모든 라이브러리를 가져와 개발 환경을 준비합니다.

In [ ]:
# ============================================================
# 라이브러리 임포트 및 환경 설정
# ============================================================
# boto3: AWS SDK for Python - AWS 서비스 호출을 위한 핵심 라이브러리
import boto3
import json
import uuid  # 고유한 세션 ID 생성용
import time  # 비동기 리소스 대기용
import requests  # Cognito 토큰 엔드포인트 HTTP 호출용
from boto3.session import Session

# AgentCore 임포트
# MemoryClient: AgentCore Memory 서비스와 통신하는 SDK 클라이언트
# StrategyType: 메모리 전략 유형 상수 (USER_PREFERENCE, SEMANTIC 등)
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands 임포트
# Agent: 모델+도구+프롬프트를 결합하여 자율 에이전트를 생성하는 핵심 클래스
# BedrockModel: Amazon Bedrock FM을 래핑하는 모델 클래스
# MCPClient: Model Context Protocol로 원격 도구 서버와 통신하는 클라이언트
# 후크 관련: 에이전트 라이프사이클 이벤트에 커스텀 로직을 주입하는 시스템
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from mcp.client.streamable_http import streamablehttp_client

# 로컬 도구 - @tool 데코레이터로 정의된 고객 지원 전용 함수들
# SYSTEM_PROMPT: 에이전트 역할/행동 지침, MODEL_ID: 사용할 Bedrock 모델 식별자
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
# SSM Parameter Store 유틸리티 - 민감한 설정값을 안전하게 저장/조회
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# 환경 설정
boto_session = Session()
REGION = "us-east-1"  # 모든 AWS 리소스가 배포된 리전
CUSTOMER_ID = "customer_001"  # 시뮬레이션할 고객의 고유 식별자
SESSION_ID = str(uuid.uuid4())  # 현재 대화 세션을 구분하는 UUID

print("✅ Libraries imported successfully!")


에이전트를 만들기 전에 고객 지원 기능을 강화할 로컬 도구를 살펴봅니다. `lab_helpers/lab1_strands_agent.py` 페이지를 열고 검토하여 다음을 이해하십시오.

- 도구는 `@tool` 데코레이터를 사용하여 이 파일에서 로컬로 정의됩니다.
- 4가지 도구 함수와 용도:
  - get_product_info(): 제품 정보 가져오기
  - get_return_policy(): 특정 제품의 반품 정책 가져오기
  - get_technical_support(): 기술 지원 지침 제공
  - web_search(): 웹에서 업데이트된 정보 검색
- 모의 데이터를 사용하는 방식(실제 데이터베이스 및 API 시뮬레이션)
- 에이전트 동작을 정의하는 시스템 프롬프트

쿼리 이해부터 작업 실행에 이르기까지 핵심 AI 기능을 보여주는 기본 고객 지원 에이전트를 만듭니다. 이 에이전트는 다음을 결합합니다.

- **파운데이션 모델**: 추론과 의사 결정을 지원하는 “뇌”
- **시스템 프롬프트**: 에이전트의 성격과 서비스 기준을 정의하는 행동 지침
- **전문 도구**: 4가지 로컬 도구(제품 정보, 반품 정책, 기술 지원, 웹 검색)

에이전트에게 전화를 걸면 다음과 같은 절차가 진행됩니다.
1. **쿼리 분석**: 에이전트가 고객의 질문을 분석합니다.
2. **도구 선택**: 에이전트가 사용할 도구(있는 경우)를 결정합니다.
3. **도구 실행**: 에이전트가 올바른 파라미터를 사용하여 적절한 도구를 호출합니다.
4. **응답 합성**: 에이전트는 도구 결과와 지식을 결합하여 유용한 응답을 생성합니다.
5. **품질 검사**: 에이전트는 응답이 시스템 프롬프트의 표준을 충족하는지 확인합니다.

In [ ]:
# ============================================================
# 기본 고객 지원 에이전트 생성
# ============================================================
# BedrockModel: Bedrock FM을 에이전트의 추론 엔진으로 래핑
#   - model_id: 사용할 모델 (예: Claude 3)
#   - temperature: 응답 창의성 (0.0=결정적, 1.0=창의적). 고객 지원은 일관성이 중요하므로 낮게 설정
#   - region_name: Bedrock 서비스 리전
model = BedrockModel(model_id="______", temperature=______, region_name="______")

# Agent: 모델 + 도구 + 시스템 프롬프트를 결합하여 자율 에이전트 구성
#   - model: 추론 엔진 (Bedrock FM)
#   - tools: 에이전트가 호출 가능한 도구 리스트 (@tool 데코레이터 함수)
#   - system_prompt: 에이전트의 역할, 성격, 응답 규칙 정의
basic_agent = Agent(
    model=model,
    tools=[______, ______, ______, ______],
    system_prompt=______
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")


기본 에이전트를 테스트하여 고객 쿼리를 처리하고 도구를 사용하는 방법을 확인합니다.

In [ ]:
# ============================================================
# 기본 에이전트 기능 테스트
# ============================================================
# Agent를 함수처럼 호출하면 자동으로 추론-도구실행-응답 사이클이 실행됨
# 내부 동작: 쿼리 분석 → 도구 선택(get_return_policy) → 도구 실행 → 응답 합성
print("💬 Testing basic agent...\n")
response = ______("______")
print("\n" + "="*50 + "\n")


이 초기 프로토타입에는 이후 태스크에서 해결해야 할 몇 가지 제한 사항이 있습니다.

- **영구 메모리 없음** - 에이전트가 이전 세션의 고객 기록과 기본 설정을 잊음
- **로컬 도구만 해당** - 공유 또는 엔터프라이즈급 도구 통합 없음  
- **자격 증명 관리 없음** - 특정 사용자를 대신하여 활동할 수 없음

### 태스크 1.2: 메모리를 활용한 에이전트 강화

소중한 고객이 최근 주문과 관련된 문제로 지원 팀에 문의합니다. 자신의 기본 설정을 설명하고, 불만을 털어놓고, 에이전트와 협력하여 문제를 해결합니다. 3주 후, 관련 질문으로 지원 팀에 다시 연락합니다. 하지만 이제 에이전트는 현재 대화 세션만 기억하고 이전 세션은 기억하지 않기 때문에 이 고객은 기본 설정, 기록, 컨텍스트 등 모든 것을 반복해야 합니다. 그 결과는 다음과 같습니다.
- **불만스러운 고객**: 정보를 반복해서 제공해야 함
- **비효율적인 지원**: 이전 상호 작용을 기반으로 구축 불가능
- **고객 만족도 저하**: 너무 광범위하고 일반적인 응답으로 인함

Amazon Bedrock AgentCore Memory는 시간이 지나도 AI 에이전트가 컨텍스트를 유지하고, 중요한 사실을 기억하고, 일관되고 개인화된 경험을 제공할 수 있는 관리형 서비스를 제공하여 이러한 한계를 극복합니다. AgentCore Memory는 다음 2가지 수준에서 작동합니다.
- **단기 메모리**: 현재 대화 컨텍스트와 세션 기반 정보(Stands Agent 프레임워크에서 자동으로 처리)
- **장기 메모리**: 사실, 기본 설정, 요약을 포함하여 여러 대화에서 추출된 영구 정보(USER_PREFERENCE 및 SEMANTIC 전략을 사용하여 AgentCore Memory 서비스를 통해 구현)

프로토타입을 다음 예시를 수행할 수 있는 고객 인식 어시스턴트로 전환합니다.
- **“반가워요, 사라!”** - 재방문 고객을 즉시 인식합니다.
- **“지난 달 발생한 노트북 문제에 대한 후속 조사”** - 관련 대화를 원활하게 연결합니다.
- **“구매 내역을 바탕으로 제가 추천하는 것은”** - 맞춤형 제안

이 태스크를 완료하면 에이전트는 통합 메모리 기능을 갖춘 다음과 같은 아키텍처를 갖게 됩니다.

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_ko_kr.png" width="75%"/>
</div>

**이미지 설명: 지속적인 고객 컨텍스트 및 개인화를 위해 AgentCore Memory로 향상된 에이전트**

**메모리 전략 구성**: 2가지 지능형 전략을 결합하여 메모리 리소스를 생성합니다.

| 전략 유형 | 목적 | 고객 혜택 |
|---------------|---------|------------------|
| USER_PREFERENCE | 고객 기본 설정 및 행동 | “...를 선호하셨던 걸로 기억합니다.” |
| SEMANTIC | 사실 정보 및 맥락 | “이전 문제에 대해...” |

네임스페이스를 사용하는 AgentCore Memory는 actorId를 사용하여 장기 메모리 메시지를 논리적으로 그룹화합니다.
- `support/customer/{actorId}/preferences`: 사용자 기본 설정 메모리 전략의 경우
- `support/customer/{actorId}/semantic`: 시맨틱 메모리 전략의 경우

In [ ]:
# ============================================================
# AgentCore Memory 클라이언트 초기화 및 메모리 리소스 생성
# ============================================================
# MemoryClient: AgentCore Memory 서비스와 통신하는 SDK 클라이언트
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    """메모리 리소스를 가져오거나 새로 생성하는 함수 (멱등성 보장)"""
    try:
        # SSM에서 기존 메모리 ID를 조회하여 재사용 (중복 생성 방지)
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # 기존 리소스가 없으면 두 가지 전략으로 새 메모리 리소스 생성
        strategies = [
            {
                # USER_PREFERENCE: 고객 선호도/행동 패턴 자동 추출
                # 예: "ThinkPad 선호", "16GB RAM 이상 원함"
                StrategyType.______.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC: 대화에서 사실 정보를 추출하여 벡터 임베딩으로 저장
                # 예: "MacBook Pro 과열 문제", "주문 #MB-78432 보증 기간 내"
                StrategyType.______.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # create_memory_and_wait: 벡터 인덱스+추출 파이프라인 프로비저닝 후 READY까지 대기
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # 90일 후 메모리 이벤트 자동 만료 (데이터 보존 정책)
        )
        memory_id = response["id"]
        # 향후 세션에서 재사용할 수 있도록 SSM에 저장
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")


이전에 지원 팀과 상호 작용한 적이 있는 ‘customer_001’이라는 재방문 고객을 시뮬레이션해 보십시오. AgentCore Memory가 어떻게 개별 대화를 풍부하고 지속적인 고객 인사이트로 자동 변환하는지 보여줍니다. 기존 고객 상호 작용을 로드하고, AgentCore Memory가 자동으로 이를 장기적인 고객 인사이트로 전환하는 과정을 확인하십시오.

In [ ]:
# ============================================================
# 이전 고객 상호 작용 시드 (테스트용 시뮬레이션)
# ============================================================
# 실제 환경에서는 CRM/채팅 로그에서 데이터가 유입되지만,
# 여기서는 테스트를 위해 이전 대화를 수동으로 주입합니다.
# 각 튜플: (메시지 내용, 역할) - "USER" 또는 "ASSISTANT"
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    # create_event: 대화 이벤트를 AgentCore Memory에 저장
    # 저장 후 비동기적으로 USER_PREFERENCE/SEMANTIC 전략이 자동 처리됨
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,  # 네임스페이스의 {actorId}에 매핑
        session_id="previous_session",  # 세션별 메시지 그룹화
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")


Strands Agents는 강력한 후크 시스템을 지원합니다. 이 시스템에서 구성 요소는 에이전트 동작에 반응하거나 에이전트 동작을 수정할 수 있으며 이때 이벤트 콜백은 강형식(strongly-typed)으로 처리됩니다. 따라서 수동 개입 없이 메모리 작업이 자동으로 수행됩니다.

고객이 에이전트와 상호 작용을 할 때마다 자동으로 다음이 수행됩니다.
- 이전 상호 작용과 기본 설정을 기반으로 **대화를 개인화**합니다.
- **메모리에 새로운 상호 작용을 추가**하여 향후 개인화를 지속적으로 개선합니다.

후크 통합의 결과:
- **응답 전**: 관련 고객 상황 및 기본 설정을 자동으로 검색
- **응답 후**: 새 상호 작용을 AgentCore Memory에 자동으로 저장

메모리 후크를 사용하여 자동 고객 컨텍스트를 활성화합니다.

In [ ]:
# ============================================================
# 메모리 후크 클래스 - 에이전트 라이프사이클에 메모리 로직 자동 주입
# ============================================================
# HookProvider: Strands의 후크 인터페이스
# register_hooks()를 구현하여 에이전트에 이벤트 핸들러를 등록
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        # get_memory_strategies: 메모리에 설정된 전략 목록 조회
        # 각 전략 타입(USER_PREFERENCE, SEMANTIC)과 네임스페이스를 매핑
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        """MessageAddedEvent 후크: 사용자 메시지 추가 시 실행.
        에이전트 응답 전에 장기 메모리에서 관련 컨텍스트를 검색하여
        사용자 메시지 앞에 주입 → 개인화된 응답 생성 유도"""
        messages = event.agent.messages
        # toolResult 포함 메시지는 도구 실행 결과이므로 건너뜀
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # 각 전략 네임스페이스에서 시맨틱 검색 수행
                # retrieve_memories: 쿼리와 의미적으로 유사한 메모리를 벡터 검색
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.______,
                        namespace=namespace.format(actorId=self.______),
                        query=______,  # TODO: 현재 사용자 질문 변수를 전달하세요
                        top_k=______,  # TODO: 반환할 최대 결과 수를 지정하세요
                    )
                    # 검색된 메모리에서 텍스트 콘텐츠 추출
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # 검색된 컨텍스트를 사용자 메시지 앞에 주입
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        """AfterInvocationEvent 후크: 에이전트 응답 완료 후 실행.
        새 대화를 AgentCore Memory에 저장 → 비동기적으로 장기 메모리로 변환됨"""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # 메시지를 역순 탐색하여 가장 최근 사용자-어시스턴트 쌍 찾기
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # 대화 쌍을 AgentCore Memory에 이벤트로 저장
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.______,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """HookProvider 인터페이스 구현: 에이전트에 후크 등록
        - MessageAddedEvent → 응답 전 컨텍스트 검색
        - AfterInvocationEvent → 응답 후 상호 작용 저장"""
        registry.add_callback(______, self.retrieve_customer_context)
        registry.add_callback(______, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")


메모리 향상 에이전트를 만들고 테스트하여 고객 컨텍스트를 검색하고 응답을 개인화하는 방법을 확인하십시오.

In [ ]:
# ============================================================
# 메모리 향상 에이전트 생성
# ============================================================
# 후크 인스턴스 생성 → 에이전트에 연결하면 모든 대화에서 자동으로:
#   1. 응답 전: 장기 메모리에서 관련 고객 컨텍스트 검색
#   2. 응답 후: 새 대화 내용을 메모리에 저장
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# hooks 파라미터: HookProvider 리스트를 전달하여 에이전트 라이프사이클에 연결
memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],  # 메모리 자동화 후크 등록
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")


In [ ]:
# ============================================================
# 메모리 처리 대기 및 테스트
# ============================================================
# AgentCore Memory는 이벤트 저장 후 비동기적으로 처리:
#   - USER_PREFERENCE: 선호도를 구조화된 형태로 추출
#   - SEMANTIC: 사실 정보를 벡터 임베딩으로 변환/인덱싱
# 처리에 약 60-90초 소요
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# 메모리 리콜 테스트 - 이전 대화에서 추출된 선호도가 반환되는지 확인
# 기대 결과: ThinkPad, 16GB RAM, Linux 호환성 등의 선호도 정보
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")


### 태스크 1.3: Gateway 통합 및 AgentCore Identity를 통한 규모 조정

메모리를 두고, 강력한 도구에 집중하고 그 효과를 확장하십시오. 훌륭한 에이전트에게는 내부와 외부 고객 모두의 업무를 처리할 수 있는 독점 및 서드 파티 API와 데이터를 최대한 활용할 수 있는 도구가 필요합니다. 그러나 에이전트 도구 구축, 보안, 규모 조정은 어렵기 때문에 고객이 에이전트 프로토타입에서 실제 에이전트 비즈니스 가치로 전환하는 데 상당한 걸림돌이 되고 있습니다. AgentCore Gateway는 AI 에이전트가 통합 **MCP(Model Context Protocol: 모델 컨텍스트 프로토콜)** 엔드포인트를 통해 실제 도구를 검색, 인증하고 호출하도록 지원하는 연결 계층 역할을 합니다. 이는 수백 개의 API, 리소스 및 도구를 관리하는 기업에 매우 중요합니다.

주요 이점:
- 인프라 관리가 필요 없는 **완전관리형 MCP 서버** 솔루션
- **기존 API**와 Lambda 함수 통합
- 다양한 도구에 걸쳐 **일관된 인터페이스**
- **보안 인증** 및 권한 부여
- **시맨틱 도구 검색** 및 선택

##### 빌드할 내용:

**도구 중앙 집중화 및 재사용성:**
- 웹 검색을 로컬 도구에서 중앙 집중식 AgentCore Gateway로 마이그레이션
- 기존 엔터프라이즈 Lambda 함수 통합(보증 확인)
- 여러 에이전트 유형이 액세스할 수 있는 공유 도구 인프라 만들기

**엔터프라이즈급 보안:**
- Cognito 통합을 통한 JWT 기반 인증 구현
- 게이트웨이 액세스를 위한 보안 인바운드 인증 구성
- 도구 사용에 대한 자격 증명 기반 액세스 제어 설정

이를 통해 도구를 중앙에서 관리하고 여러 에이전트 유형에서 재사용할 수 있는 확장 가능한 기반을 구축하여 코드 중복을 제거하고 유지 관리를 간소화합니다.

AgentCore Identity도 이 프로세스에 참여합니다. 이를 통해 AI 에이전트는 AWS 리소스에 안전하게 액세스하고 Amazon Cognito와 협력하여 인바운드 발신자 인증을 처리할 수 있습니다. 또한 이 실습에서는 Agentcore Identity의 이 기능을 사용하지 않지만 AI 에이전트는 아웃바운드 인증을 사용하여 서드 파티 도구와 서비스에 안전하게 액세스할 수 있습니다.

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_ko_kr.png" width="75%"/>
</div>

이 태스크를 완료하면 에이전트는 통합 게이트웨이 기능을 갖춘 다음과 같은 아키텍처를 갖게 됩니다.

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_ko_kr.png" width="75%"/>
</div>

**이미지 설명: 안전한 중앙 집중식 도구 관리 및 엔터프라이즈 통합을 위해 AgentCore Gateway로 향상된 에이전트**

AgentCore Gateway를 생성하여 Lambda 함수를 MCP 호환 엔드포인트로 노출합니다. 도구를 호출할 권한이 있는 발신자를 확인하려면 MCP 서버의 표준인 OAuth 인증을 사용하여 **인바운드 인증**을 구성하십시오.

### 게이트웨이 인증에 대한 이해

AgentCore Gateway는 **JWT 토큰과 함께 OAuth 2.0**을 사용하여 도구에 대한 액세스를 보호합니다. 그러면 승인되지 않은 애플리케이션이 Lambda 함수를 호출하는 것을 방지할 수 있습니다.

**주요 개념:**

1. **인증 공급자**: Amazon Cognito는 자격 증명을 관리하고 토큰을 발행합니다.
2. **클라이언트 자격 증명**: 에이전트는 client_id 및 client_secret(예: 애플리케이션의 사용자 이름 및 암호)를 사용합니다.
3. **JWT 토큰**: 에이전트가 승인되었음을 증명하는 수명이 짧은 토큰
4. **허용된 클라이언트**: 게이트웨이에 액세스할 수 있는 클라이언트 ID의 허용 목록

**작동 방식:**
```
에이전트 → Cognito: “내 client_id와 client_secret을 제출함”
Cognito → 에이전트: “당신의 JWT 액세스 토큰은 이러함”
에이전트 → Gateway: “내 토큰은 이러함”
Gateway → Cognito: “이 토큰은 유효하고 허용된 클라이언트에서 온 것입니까?”
Gateway → 에이전트: “액세스 허용됨”
```

**보안 참고 사항**: 이러한 자격 증명은 사용자를 위해 사전 생성되었으며 SSM 파라미터 스토어에 안전하게 저장되었습니다. 코드에 자격 증명을 하드 코딩해서는 안 됩니다.

In [ ]:
# ============================================================
# SSM Parameter Store에서 인증 구성 검색
# ============================================================
# CloudFormation 스택이 배포 시 이 파라미터들을 자동 생성

# Client ID: OAuth 2.0에서 애플리케이션을 고유하게 식별하는 값
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL (OIDC): 토큰 엔드포인트, 서명 키 등 OAuth 메타데이터 제공
# Gateway가 JWT 검증에 필요한 정보를 이 URL에서 자동으로 가져옴
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Gateway의 JWT 인가(Authorization) 구성
# customJWTAuthorizer: 수신 요청의 JWT 토큰 검증 방법 정의
auth_config = {
    "customJWTAuthorizer": {
        # allowedClients: 이 클라이언트 ID의 토큰만 허용 (허용 목록)
        "allowedClients": [machine_client_id],
        # discoveryUrl: JWT 서명 검증용 공개 키를 가져올 OIDC 엔드포인트
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")


### AgentCore Gateway 생성

Gateway는 에이전트와 백엔드 Lambda 함수 간의 보안 프록시 역할을 합니다. AI 에이전트를 위해 특별히 설계된 API 게이트웨이라고 생각하면 됩니다.

**우리가 만들고 있는 것:**
- **Gateway 인프라**: 핵심 Gateway 리소스
- **MCP 프로토콜**: 도구 통신을 위한 표준 프로토콜
- **JWT 인증**: 방금 생성한 인증 구성 사용
- **IAM 역할**: Lambda 함수를 호출할 수 있는 권한

**다음은 어떻게 됩니까?**
1. Gateway 생성(이 셀)
2. 도구 정의가 포함된 Lambda 대상 추가(다음 셀)
   - 하나의 Lambda 함수가 여러 도구 처리: `check_warranty_status`, `web_search`
   - Gateway는 도구 이름을 Lambda에 전달하고 Lambda는 적절한 핸들러로 라우팅
3. 에이전트를 Gateway에 연결

**참고**: CloudFormation 템플릿은 여러 도구 작업을 처리할 수 있는 단일 Lambda 함수(`CustomerSupportLambda`)를 배포했습니다. 각 도구에 대해 별도의 Lambda 함수를 배포하는 방식보다 더 효율적입니다.

In [ ]:
# ============================================================
# AgentCore Gateway 생성
# ============================================================
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

# bedrock-agentcore-control: AgentCore 리소스 생성/관리 컨트롤 플레인 API
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    # create_gateway: AgentCore Gateway 리소스 생성
    #   - protocolType: "MCP" - AI 에이전트 도구 통신 표준
    #   - authorizerType: "CUSTOM_JWT" - JWT 토큰 기반 인바운드 인가
    #   - authorizerConfiguration: JWT 검증 규칙 (허용 클라이언트, Discovery URL)
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="______",  # TODO: AI 에이전트 도구 통신 표준 프로토콜
        authorizerType="______",  # TODO: JWT 토큰 기반 인가 방식
        authorizerConfiguration=auth_config,
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]  # MCP 연결용 HTTP 엔드포인트
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Gateway는 비동기 프로비저닝 → READY 상태까지 폴링
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # ConflictException: 동일 이름 Gateway가 이미 존재 → 재사용 (멱등성)
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")


AgentCore Gateway는 호출할 도구의 이름으로 Lambda 컨텍스트를 채우고, 도구에 전달된 파라미터는 Lambda 이벤트에 제공됩니다. 이를 통해 여러 에이전트에서 재사용할 수 있는 기존 엔터프라이즈 Lambda 함수(이 경우 `AgentCoreLab-CustomerSupportLambda`)를 통합할 수 있습니다.

API 사양을 사용하여 Lambda 함수를 게이트웨이 대상으로 추가합니다.

In [ ]:
# ============================================================
# Gateway Target 생성 - Lambda 함수를 MCP 도구로 노출
# ============================================================
# 도구 스키마: Gateway에 등록할 도구의 JSON Schema 정의
# Gateway는 이 스키마를 MCP tools/list 응답으로 에이전트에 전달
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Gateway Target 구성: Lambda를 MCP 백엔드로 연결
# mcp.lambda: Gateway가 MCP 도구 호출을 Lambda로 라우팅
#   - lambdaArn: 호출할 Lambda ARN
#   - toolSchema.inlinePayload: 도구 스키마를 인라인으로 제공
lambda_target_config = {
    "______": {
        "______": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"______": api_spec},  # TODO: 도구 스키마 인라인 제공 키
        }
    }
}

try:
    # credentialProviderType: "GATEWAY_IAM_ROLE" - Gateway 자체 IAM 역할로 Lambda 호출
    # (에이전트 자격 증명이 아닌 Gateway 역할 사용 → 보안 경계 분리)
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "______"  # TODO: Gateway 자체 IAM 역할 사용}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")


Cognito의 인증 토큰을 Strands SDK의 MCP Client에 통합하여 안전한 MCP 연결을 생성합니다.

인증된 MCP 클라이언트를 생성하여 게이트웨이 도구에 액세스합니다.

In [ ]:
# ============================================================
# OAuth 2.0 Client Credentials Flow로 Gateway 인증 토큰 획득
# ============================================================
def get_cognito_client_secret():
    """Cognito User Pool에서 앱 클라이언트의 시크릿 키 조회"""
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    """OAuth 2.0 Client Credentials Grant로 액세스 토큰 발급.
    서버-투-서버(M2M) 인증: 사용자 개입 없이 앱 자체 자격 증명으로 토큰 발급"""
    headers = {"Content-Type": "______"  # TODO: 폼 데이터 전송용 Content-Type}
    data = {
        "grant_type": "______",  # TODO: M2M 인증용 OAuth 2.0 그랜트 타입
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Cognito 토큰 엔드포인트에 POST → JWT 형식 액세스 토큰 발급
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# JWT 액세스 토큰 발급
token_response = get_oauth_token()
access_token = token_response['access_token']

# MCPClient: Strands SDK의 MCP 프로토콜 클라이언트
# streamablehttp_client: HTTP/SSE 기반 MCP 전송 계층
# Bearer 토큰을 Authorization 헤더에 포함하여 Gateway 인증
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway_url,
        headers={"______": f"______ {access_token}"},  # TODO: HTTP 인증 헤더와 스킴
    )
)

print(f"✅ MCP client configured for gateway: {gateway_url}")


메모리 후크 + 로컬 도구 + 게이트웨이 도구 등 모든 것을 결합합니다. 이를 통해 일부 도구는 속도와 단순성을 위해 로컬에 유지되고, 다른 도구는 재사용성과 엔터프라이즈 통합을 위해 게이트웨이를 통해 중앙 집중화되는 하이브리드 아키텍처가 구축됩니다.

이 접근 방식은 여러 에이전트 간의 코드 중복을 제거하고 도구 업데이트를 중앙 집중식으로 관리합니다.

In [ ]:
# ============================================================
# 최종 통합 에이전트: 메모리 + 로컬 도구 + Gateway 도구
# ============================================================
# 하이브리드 아키텍처: 로컬(빠른 응답) + Gateway(중앙 관리, 재사용)
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# MCP 클라이언트 시작: Gateway와 Streamable HTTP 연결 수립 (initialize 핸드셰이크)
mcp_client.start()
# list_tools_sync: Gateway에 등록된 도구 목록을 MCP tools/list로 조회
gateway_tools = mcp_client.list_tools_sync()

# 로컬 도구 + Gateway 도구를 하나의 리스트로 결합
# Agent는 도구 출처(로컬/원격)를 구분하지 않고 동일하게 사용
all_tools = [
    get_product_info,      # 로컬: 제품 정보 조회
    get_return_policy,     # 로컬: 반품 정책 조회
    get_technical_support, # 로컬: 기술 지원 안내
] + gateway_tools          # 원격: web_search + check_warranty_status (Lambda)

# 최종 에이전트: 모든 기능 통합
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # 로컬(3) + Gateway(2) = 총 5개 도구
    hooks=[memory_hooks],      # 메모리 자동화 (컨텍스트 검색 + 저장)
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")


메모리와 게이트웨이 기능으로 에이전트를 테스트합니다. 에이전트가 메모리를 통해 고객 컨텍스트를 유지하면서 로컬 도구와 중앙 집중식 게이트웨이 도구를 모두 원활하게 사용할 수 있는지 확인합니다.

테스트 시나리오에는 보증 확인, 웹 검색, 메모리 + 게이트웨이 통합 기능이 포함됩니다.

In [ ]:
# Gateway 웹 검색 도구 테스트
# 흐름: Agent → MCP 요청 → Gateway(JWT 검증) → Lambda(web_search) → 응답
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")


In [ ]:
# Gateway 보증 확인 도구 테스트
# check_warranty_status: 시리얼 번호로 보증 상태 조회 (Lambda 핸들러)
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")


In [ ]:
# 메모리 + Gateway 통합 테스트
# 1. 메모리: "gaming headphones again"에서 이전 대화(FPS 게임용 헤드폰) 컨텍스트 검색
# 2. Gateway: web_search 도구로 최신 리뷰 검색
# 에이전트가 개인화된 추천 + 최신 정보를 결합하여 응답
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")


## 다음 단계

🎉 **축하합니다!** 노트북 연습을 완료했습니다.

다음을 성공적으로 수행했습니다.
- Strands를 이용한 기본 AI 에이전트 프로토타입을 제작했습니다.
- 지속적인 고객 컨텍스트를 위해 AgentCore Memory로 개선했습니다.
- 안전한 중앙 집중식 도구 공유를 위해 AgentCore Gateway를 통합했습니다.
- 프로덕션에 바로 사용할 수 있는 완벽한 고객 지원 시스템인지 테스트했습니다.

### 다음 단계는 무엇입니까?

1. **이 노트북 파일을 닫습니다.**
2. **실습 지침으로 돌아갑니다.**
3. **태스크 2를 계속 진행하여** AgentCore 대시보드를 탐색하고 리소스가 어떻게 작동하는지 확인하십시오.
